In [2]:
import pandas as pd
from xgboost_distribution import XGBDistribution

# 載入先前保存的模型
loaded_model =XGBDistribution()
loaded_model.load_model('xgboost_photoz_model.json')

In [ ]:
MAG = 'Kron'
ps1filename = pd.read_csv('../data/merged_ps1.csv')
bands = ['g', 'r', 'i', 'z', 'y']
for b in bands:
    ps1filename = ps1filename[ps1filename[f'{b}Mean{MAG}Mag']!= -999.0]
    ps1filename = ps1filename[ps1filename[f'{b}Mean{MAG}MagErr']!= -999.0]
ps1filename = ps1filename[((ps1filename['rMeanPSFMag'] - ps1filename['rMeanKronMag']) > 0.3)]

In [4]:
from xgboost_distribution import XGBDistribution
from sklearn.model_selection import train_test_split
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
from dustmaps.csfd import CSFDQuery
import astropy.units as u

# 1. 準備您的數據 (此部分維持原樣)
ml_data = ps1filename.copy()

def extract_features(df):
    """
    計算去紅化星等並提取 5 維度的特徵空間。
    """
    df = df.copy()
    bands = ['g', 'r', 'i', 'z', 'y']
    
    # 1. 去紅化計算
    csfd = CSFDQuery()
    coords = SkyCoord(ra = df['raMean'], dec = df['decMean'], unit = (u.deg, u.deg), frame = 'icrs')
    ebv = csfd(coords)
    coefficient = [3.634, 2.241, 1.568,  1.258, 1.074]
    extinction = [i*ebv for i in coefficient]
    for b, extinc in zip(bands, extinction):
        if f'{b}Mean{MAG}Mag' in df.columns:
            df[f'dered_{b}'] = df[f'{b}Mean{MAG}Mag'] - extinc
        else:
            raise ValueError(f"缺少 {b} 波段的資料")
            
    # 2. 計算 5D 測光特徵空間
    features = pd.DataFrame()
    features['mag_r'] = df['dered_r']
    features['g-r'] = df['dered_g'] - df['dered_r']
    features['r-i'] = df['dered_r'] - df['dered_i']
    features['i-z'] = df['dered_i'] - df['dered_z']
    features['z-y'] = df['dered_z'] - df['dered_y']
    # for b in bands:
    #    features[f'{b}_err'] = df[f'{b}MeanApMagErr']

    return df, features

# 計算特徵顏色
ml_data, ml_feature = extract_features(ml_data)

In [5]:
from astropy.cosmology import Planck18

preds = loaded_model.predict(ml_feature)
ml_data['z_phot'] = preds.loc

from scipy.stats import norm

# 計算連續 PDF 的數值 (畫圖用)
# pdf_values = laplace.pdf(z_grid, loc=mean_val, scale=std_val)

# 如果要用剛才超快速的 CDF 寫法算 Odds：
d_array = 0.03 * (1 + preds.loc)
ml_data['odds'] = norm.cdf(preds.loc + d_array, loc=preds.loc, scale=preds.scale) - norm.cdf(preds.loc - d_array, loc=preds.loc, scale=preds.scale)
ml_data['z_phot_err'] = preds.scale

cosmo = Planck18
def calculate_physical_parameters(df):
    # 避免紅移為 0 導致發光距離與對數運算錯誤，設定下限
    z_safe = np.clip(df['z_phot'].values, 1e-4, None)
    
    # 1. 計算發光距離 D_L (單位 Mpc) 並轉換為 pc
    df['D_L_Mpc'] = cosmo.comoving_distance(z_safe).value
    df['D_L_pc'] = df['D_L_Mpc'] * 1e6
    
    # 2. 計算絕對星等 (Absolute Magnitude)
    # 公式: M_r = m_r - 5*log10(D_L/10) - K_correction
    # 備註: 這裡暫時假設 K-correction = 0
    distance_modulus = 5 * np.log10(df['D_L_pc'] / 10)
    df['Absolute_Mag_r'] = df['dered_r'] - distance_modulus
    
    # 3. 計算相對通量 (Flux)
    # 基於標準 SDSS AB 零點系統轉換: Flux = 10 ** ((8.90 - m) / 2.5)
    df['Calculated_Flux'] = 10 ** ((8.90 - df['dered_r']) / 2.5)
    
    return df
ml_data = calculate_physical_parameters(ml_data)

In [6]:
def data_export(df):
    export_columns = [
        'objID', 'z_phot', 'z_phot_err','odds', 
        'D_L_Mpc', 'Absolute_Mag_r', 'Calculated_Flux'
    ]

    final_catalog = df[export_columns].copy()
    return final_catalog

In [7]:
ml_data = data_export(ml_data)
ml_data.to_csv('ps1_estimate_xgb.csv')
ml_data[ml_data['odds'] > 0.82].to_csv('ps1_estimate_filtered_xgb.csv')